In [ ]:
import os

def parse_guitar_log(filepath):
    """
    指定されたファイルを読み込み、運指データを抽出する関数
    休符(R)は除外します。
    """
    data = []
    is_data_section = False
    
    # ファイルが存在するか確認
    if not os.path.exists(filepath):
        print(f"エラー: ファイルが見つかりません -> {filepath}")
        return []

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()
    except UnicodeDecodeError:
        # utf-8で読めない場合はshift_jisなどを試行
        with open(filepath, 'r', encoding='shift_jis') as f:
            lines = f.readlines()
            
    for line in lines:
        line = line.strip()
        
        # データセクションの開始を検知（ヘッダー行を探す）
        if "Note" in line and "String" in line and "Fret" in line:
            is_data_section = True
            continue
        
        # データセクションの終了条件（コスト内訳などが来たら終了）
        if is_data_section and ("---" in line or "Cost Breakdown" in line):
            break
            
        if is_data_section and line:
            parts = line.split()
            
            # データ行としての要件を満たしているか確認
            if len(parts) >= 5:
                note = parts[0]
                
                # 休符(R)は除外する
                if note == 'R':
                    continue
                
                # 必要な情報を抽出 
                entry = {
                    'note': parts[0],
                    'string': parts[1],
                    'fret': parts[2],
                    'hand': parts[3],  # ポジション
                    'finger': parts[4]
                }
                data.append(entry)
                
    return data

def count_finger_usage(data):
    """
    各指の使用回数をカウントする関数
    指: 1=人差し指, 2=中指, 3=薬指, 4=小指, x=開放弦
    """
    finger_counts = {'1': 0, '2': 0, '3': 0, '4': 0, 'x': 0}
    
    for entry in data:
        finger = entry['finger']
        if finger in finger_counts:
            finger_counts[finger] += 1
        else:
            # 想定外の値があればカウント
            finger_counts[finger] = finger_counts.get(finger, 0) + 1
    
    return finger_counts

def calculate_similarity(file1, file2):
    """
    2つのファイルを比較して差異の割合を計算・表示する関数
    """
    print(f"比較開始: \n file1: {file1}\n file2: {file2}\n")
    
    data1 = parse_guitar_log(file1)
    data2 = parse_guitar_log(file2)
    
    # ノート数のチェック
    len1 = len(data1)
    len2 = len(data2)
    
    if len1 == 0 or len2 == 0:
        print("処理中断: 有効なデータが読み込めませんでした。")
        return

    print(f"ファイル1の有効ノート数: {len1}")
    print(f"ファイル2の有効ノート数: {len2}")
    
    if len1 != len2:
        print("警告: 2つのファイルのノート数（休符除く）が一致しません。")
        print("短い方の長さに合わせて先頭から比較します。")
    
    compare_len = min(len1, len2)
    
    diff_string_fret_count = 0   # ① 弦・フレットが異なるノートの数
    diff_finger_count = 0        # ② 弦・フレットが同じ中で指が異なるノートの数
    diff_position_count = 0      # ③ ポジションが異なる数
    
    for i in range(compare_len):
        d1 = data1[i]
        d2 = data2[i]
        
        # 弦・フレットの一致確認
        is_pos_match = (d1['string'] == d2['string']) and (d1['fret'] == d2['fret'])
        
        if not is_pos_match:
            # ① 弦・フレットが異なる
            diff_string_fret_count += 1
        else:
            # 弦・フレットが同じ場合のみ、指の差異をカウント
            if d1['finger'] != d2['finger']:
                # ② 弦・フレットが同じ中で指が異なる
                diff_finger_count += 1
        
        # ③ ポジションの差異
        if d1['hand'] != d2['hand']:
            diff_position_count += 1

    # 割合の計算
    ratio_string_fret = (diff_string_fret_count / compare_len) * 100
    ratio_finger = (diff_finger_count / compare_len) * 100
    ratio_position = (diff_position_count / compare_len) * 100
    
    print("-" * 50)
    print(f"比較対象ノート数: {compare_len}")
    print(f"① 弦・フレットが異なるノート    : {ratio_string_fret:.2f}% ({diff_string_fret_count}/{compare_len})")
    print(f"② 弦・フレット同じで指が異なる  : {ratio_finger:.2f}% ({diff_finger_count}/{compare_len})")
    print(f"③ ポジションが異なるノート      : {ratio_position:.2f}% ({diff_position_count}/{compare_len})")
    print("-" * 50)
    
    # --- 各指の使用率 ---
    finger_counts1 = count_finger_usage(data1)
    finger_counts2 = count_finger_usage(data2)
    
    finger_names = {
        '1': '人差し指',
        '2': '中指',
        '3': '薬指',
        '4': '小指',
        'x': '開放弦'
    }
    
    print("\n【各指の使用率】")
    print(f"{'指':<10} {'file1':>12} {'file2':>12} {'変化率':>12}")
    print("-" * 50)
    
    for finger in ['1', '2', '3', '4', 'x']:
        count1 = finger_counts1.get(finger, 0)
        count2 = finger_counts2.get(finger, 0)
        
        rate1 = (count1 / len1) * 100 if len1 > 0 else 0
        rate2 = (count2 / len2) * 100 if len2 > 0 else 0
        
        # 変化率 (file2 - file1)
        change = rate2 - rate1
        change_str = f"{change:+.2f}%" if change != 0 else "±0.00%"
        
        name = finger_names.get(finger, finger)
        print(f"{name:<10} {rate1:>10.2f}% {rate2:>10.2f}% {change_str:>12}")
    
    print("-" * 50)

In [ ]:
# 比較したい2つのファイルパスを指定してください
file_path_A = "complex_compare_modified_model.txt"  # 1つ目のファイル
file_path_B = "complex_compare_modified_model (8).txt"  # 2つ目のファイル（比較用）

# 関数を実行
calculate_similarity(file_path_A, file_path_B)